In [18]:
import pandas as pd
import numpy as np

In [19]:
# Load and check data
# df = pd.read_csv("../data/cleaned/dataset_mood_smartphone_cleaned.csv", parse_dates=["date"])
df = pd.read_csv("dataset_mood_smartphone_cleaned.csv", parse_dates=["date"])
print(df.dtypes)
df.head()

id                              object
date                    datetime64[ns]
activity                       float64
appCat.builtin                 float64
appCat.communication           float64
appCat.entertainment           float64
appCat.finance                 float64
appCat.game                    float64
appCat.office                  float64
appCat.other                   float64
appCat.social                  float64
appCat.travel                  float64
appCat.unknown                 float64
appCat.utilities               float64
appCat.weather                 float64
call                             int64
circumplex.arousal             float64
circumplex.valence             float64
mood                           float64
screen                         float64
sms                              int64
dtype: object


,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-17,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2,-0.245213,0.755012,7.075177,0.0,0
1,AS14.01,2014-02-18,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1,-0.245213,0.755012,7.075177,0.0,0
2,AS14.01,2014-02-19,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,7,-0.245213,0.755012,7.075177,0.0,2
3,AS14.01,2014-02-20,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2,-0.245213,0.755012,7.075177,0.0,3
4,AS14.01,2014-02-21,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,-0.245213,0.755012,7.075177,0.0,1


In [20]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df.head()

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-17,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2,-0.245213,0.755012,7.075177,0.0,0
1,AS14.01,2014-02-18,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1,-0.245213,0.755012,7.075177,0.0,0
2,AS14.01,2014-02-19,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,7,-0.245213,0.755012,7.075177,0.0,2
3,AS14.01,2014-02-20,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2,-0.245213,0.755012,7.075177,0.0,3
4,AS14.01,2014-02-21,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,-0.245213,0.755012,7.075177,0.0,1


# Feature engineering

In [21]:
# Replace valence and arousal with product as 'emotion'
df['emotion'] = df['circumplex.valence'] * df['circumplex.arousal']

df["count_call_and_sms"] = ( df["call"] + df["sms"])

df = df.drop(columns=['circumplex.valence', 'circumplex.arousal', 'sms', 'call'])
df.head()

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,appCat.social,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,mood,screen,emotion,count_call_and_sms
0,AS14.01,2014-02-17,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.075177,0.0,-0.185139,2
1,AS14.01,2014-02-18,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.075177,0.0,-0.185139,1
2,AS14.01,2014-02-19,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.075177,0.0,-0.185139,9
3,AS14.01,2014-02-20,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.075177,0.0,-0.185139,5
4,AS14.01,2014-02-21,0.080856,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.075177,0.0,-0.185139,1


In [22]:
# Replace individual "appCat" features with aggregated cateories
app_cols = [c for c in df.columns if c.startswith("appCat.")]
df["total_app_use"] = df[app_cols].fillna(0).sum(axis=1)

def safe_ratio(numerator, denominator):
    return numerator.where(denominator > 0, 0.0) / denominator.where(denominator > 0, 1.0)

df["social_ratio"] = safe_ratio(
    df["appCat.social"] + df["appCat.communication"],
    df["total_app_use"]
)

df["productive_ratio"] = safe_ratio(
    df["appCat.finance"] + df["appCat.office"] + df["appCat.utilities"],
    df["total_app_use"]
)

df["passive_entertainment_ratio"] = safe_ratio(
    df["appCat.entertainment"] + df["appCat.game"],
    df["total_app_use"]
)

df["adventure_ratio"] = safe_ratio(
    df["appCat.travel"] + df["appCat.weather"],
    df["total_app_use"]
)

df["miscellaneous_ratio"] = safe_ratio(
    df["appCat.other"] + df["appCat.unknown"] + df["appCat.utilities"],
    df["total_app_use"]
)

df = df.drop(columns=app_cols)
df.head()

,id,date,activity,mood,screen,emotion,count_call_and_sms,total_app_use,social_ratio,productive_ratio,passive_entertainment_ratio,adventure_ratio,miscellaneous_ratio
0,AS14.01,2014-02-17,0.080856,7.075177,0.0,-0.185139,2,0.0,0.0,0.0,0.0,0.0,0.0
1,AS14.01,2014-02-18,0.080856,7.075177,0.0,-0.185139,1,0.0,0.0,0.0,0.0,0.0,0.0
2,AS14.01,2014-02-19,0.080856,7.075177,0.0,-0.185139,9,0.0,0.0,0.0,0.0,0.0,0.0
3,AS14.01,2014-02-20,0.080856,7.075177,0.0,-0.185139,5,0.0,0.0,0.0,0.0,0.0,0.0
4,AS14.01,2014-02-21,0.080856,7.075177,0.0,-0.185139,1,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
df[df["total_app_use"] > df["screen"]].head()

,id,date,activity,mood,screen,emotion,count_call_and_sms,total_app_use,social_ratio,productive_ratio,passive_entertainment_ratio,adventure_ratio,miscellaneous_ratio
26,AS14.01,2014-03-21,0.087240,6.2,13984.396,0.040,6,13984.396135,0.644853,0.036063,0.065777,0.032696,0.037238
33,AS14.01,2014-03-28,0.077666,6.4,4923.489,-0.540,5,4923.489040,0.279612,0.079224,0.251007,0.159729,0.106948
36,AS14.01,2014-03-31,0.070505,7.4,8762.112,0.000,6,8762.112952,0.631374,0.023765,0.171001,0.017324,0.048005
42,AS14.01,2014-04-06,0.049669,6.5,5674.088,-0.375,4,5674.088056,0.518679,0.007199,0.046952,0.000000,0.025742
50,AS14.01,2014-04-14,0.083887,7.2,9261.666,-0.720,2,9261.666086,0.561076,0.066519,0.280599,0.002353,0.071027


In [24]:
# Add last day market for TS dataset
df_ts = df.copy()
df["is_last_day"] = (df["date"] == df["id"].map(df.groupby("id")["date"].max())).astype(int)
df[df["is_last_day"] == 1].head()

,id,date,activity,mood,screen,emotion,count_call_and_sms,total_app_use,social_ratio,productive_ratio,passive_entertainment_ratio,adventure_ratio,miscellaneous_ratio,is_last_day
71,AS14.01,2014-05-05,0.007843,7.075177,1788.73400,-0.185139,0,1744.627,0.488294,0.077882,0.182569,0.0,0.077882,1
139,AS14.02,2014-04-25,0.265355,9.000000,0.00000,-1.000000,0,0.000,0.000000,0.000000,0.000000,0.0,0.000000,1
216,AS14.03,2014-05-08,0.000000,7.603901,0.00000,-0.058889,0,0.000,0.000000,0.000000,0.000000,0.0,0.000000,1
286,AS14.05,2014-05-05,0.002491,6.333334,49.03300,-0.333333,0,48.147,0.680645,0.000000,0.107068,0.0,0.212287,1
360,AS14.06,2014-05-08,0.000000,7.000000,381.99103,-1.000000,0,381.991,0.960910,0.000000,0.039090,0.0,0.000000,1


# Aggregate entries into one per individual (id)

In [25]:
df = df.sort_values(["id", "date"])

# Create 7-day rolling window
df_rolled = df.merge(df[["id", "date", "mood"]], on="id", suffixes=("_last", "_prior"))
df_rolled = df_rolled[
    (df_rolled["date_prior"] < df_rolled["date_last"]) &
    (df_rolled["date_prior"] >= df_rolled["date_last"] - pd.Timedelta(days=6))
]

# Keep only windows with the 7 entries
prior_day_counts = df_rolled.groupby(["id", "date_last"]).size().reset_index(name="prior_day_count")
valid_last_days = prior_day_counts[prior_day_counts["prior_day_count"] == 6][["id", "date_last"]]

df_last_days = valid_last_days.rename(columns={"date_last": "last_date"})

In [26]:
# Compute target (average mood) on each last day of a window
df_last_days_mood = (
    df.merge(df_last_days, left_on=["id", "date"], right_on=["id", "last_date"])
      .groupby(["id", "last_date"])
      .agg(avg_last_day_mood=("mood", "mean"))
      .reset_index()
)

In [27]:
# Compare last_date to prior days and compute decay weights
df_prior_days = df.merge(df_last_days_mood[["id", "last_date"]], on="id")
df_prior_days = df_prior_days[
    (df_prior_days["date"] < df_prior_days["last_date"]) &
    (df_prior_days["date"] >= df_prior_days["last_date"] - pd.Timedelta(days=6))
]

decay_rate = 0.1
df_prior_days["diff_days"] = (df_prior_days["last_date"] - df_prior_days["date"]).dt.days
df_prior_days["weight"] = np.exp(-decay_rate * df_prior_days["diff_days"])

value_cols = [c for c in df_prior_days.columns if c not in ["id", "date", "last_date", "diff_days", "weight"]]
for col in value_cols:
    df_prior_days[col] = df_prior_days[col] * df_prior_days["weight"]

# Apply weighted average over attributes of prior days, per (id, last_date)
df_weighted_avg = df_prior_days.groupby(["id", "last_date"]).agg(
    **{
        col: (col, lambda x: (
            x.sum() / df_prior_days.loc[x.index[x.notna()], "weight"].sum()
            if x.notna().any() else np.nan
        ))
        for col in value_cols
    }
).reset_index()

In [28]:
# Merge datasets with target and attributes
df_non_ts = df_last_days_mood.merge(df_weighted_avg, on=["id", "last_date"], how="left")
df_non_ts = df_non_ts.dropna(subset=["avg_last_day_mood"])
df_non_ts.head()
df_non_ts.to_csv("dataset_mood_smartphone_non_ts_7_day_rolling_window.csv", index=False)

In [29]:
df_ts.to_csv("../data/engineered/dataset_mood_smartphone_ts.csv", index=False)
df_non_ts.to_csv("../data/engineered/dataset_mood_smartphone_non_ts.csv", index=False)

OSError: Cannot save file into a non-existent directory: '../data/engineered'